# 20 - ¿El SSL funciona tambien en TransUNet y en Kim BiFPN-U-Net?

## La pregunta

Es la peticion 6 de Ivson, la unica de su lista que necesita GPU:

> testar SSL nos outros modelos

Y su mensaje de Slack despues de leer la v54:

> onde esta o SSL sobre todos os backbones? estou vendo SSL na Unet++ e uma
> comparacao com outros backbones supervisionados

La tesis cubre **cuatro de seis** backbones bajo Mean Teacher (linea 915):

| backbone | supervisado | MT random r=15 |
|---|---|---|
| U-Net++ | .801 | .830 ± .038 |
| U-Net | .824 | .851 ± .005 |
| FPN | .799 | .811 ± .028 |
| DeepLabV3+ | .776 | .800 ± .014 |
| **TransUNet** | **.851 ± .011** | **falta** |
| **Kim BiFPN-U-Net(T)** | **.759 ± .012** | **falta** |

Faltan justo los dos extremos: la arquitectura mas fuerte del pipeline y la unica
sin preentrenamiento ImageNet.

Jean, en la banca, predijo el resultado de una de ellas:

> esses modelos profundos vao se beneficiar mais ainda de self-supervised learning
> [...] o intuito me diz que talvez esse TransUNet te gere ainda melhores resultados
> do que a UNet++

## Como esta montado

Un solo cambio por brazo, igual que en el notebook 19: aqui lo que cambia es `arch`.
La configuracion SSL se copia **exacta** de `runs_final_v1/mean_teacher_fpn_std_matched_r15`,
que es una de las cuatro filas que ya estan en la tesis, para que la comparacion
sea con el mismo protocolo.

El SSL no depende de la arquitectura: `src/train.py` construye estudiante y profesor
con la misma fabrica (`create_model(cfg["arch"], ...)`, lineas 490 y 530). TransUNet y
Kim no traen su propio SSL, traen su propia red.

## Orden de ejecucion

La semilla 0 de las cuatro condiciones va primero. A las ~5,6 h ya se conoce la
direccion de las dos arquitecturas; a partir de ahi el notebook completa TransUNet y
luego Kim. Es reanudable: si Colab se corta, al reejecutar salta lo ya terminado.

## Lo que NO hace

No toca `runs_final_v1` ni ningun directorio existente. Escribe en `runs_ssl_backbones/`.
Los resultados van al banco de preguntas de defensa; la tesis esta congelada.


In [ ]:
# ============================================================
# SETUP - correr una vez tras cada reinicio del runtime
# No entrena nada.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!rm -rf /content/tesis-seg
!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"

import sys
sys.path.append("/content/tesis-seg")

import json, os, time, glob, statistics

from src.defaults import get_default_config, summarize_config
from src.augmentations import get_supervised_train_augmentation
from src.datasets import (build_supervised_datasets, build_dataloaders,
                          build_unlabeled_datasets)
from src.tee import tee_output
from src.models import create_model
from src.train import run_training
from src.evaluate import evaluate_checkpoint

# --- GUARDIAN 1: TransUNet necesita un checkpoint que NO viene en el repo ---
# create_model lo busca en TRANSUNET_PRETRAINED_PATH y falla con un mensaje claro
# si no esta. El .npz vive en la raiz del Drive del proyecto.
BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
os.environ["TRANSUNET_PRETRAINED_PATH"] = f"{BASE}/R50+ViT-B_16.npz"
assert os.path.isfile(os.environ["TRANSUNET_PRETRAINED_PATH"]), (
    "FALTA el checkpoint R50+ViT-B_16.npz en la raiz del proyecto. "
    "Sin el, TransUNet no se puede construir.")
print("OK: checkpoint de TransUNet encontrado.")

# --- GUARDIAN 2: el codigo clonado debe traer las dos arquitecturas ---
import inspect
from src import models as _md
_src = inspect.getsource(_md.create_model)
for _a in ("transunet", "bifpn_unet"):
    assert f'"{_a}"' in _src, f"CODIGO VIEJO: create_model no conoce {_a}."
print("OK: create_model conoce transunet y bifpn_unet.")

# --- GUARDIAN 3: dejar constancia del entorno ---
# Colab cambio de stack entre julio y septiembre de 2026. Este notebook entrena
# su propio control supervisado en esta misma sesion para cada arquitectura, de
# modo que la comparacion SSL vs supervisado no cruza entornos.
import albumentations as _alb
import segmentation_models_pytorch as _smp
ENTORNO = {"python": sys.version.split()[0], "torch": torch.__version__,
           "albumentations": _alb.__version__, "smp": _smp.__version__,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
print()
print("ENTORNO DE ESTA SESION")
for k, v in ENTORNO.items():
    print("   %-16s %s" % (k, v))
print()
print("Las cuatro filas de la tesis (l.915) se entrenaron con albumentations 1.3.1.")
print("Si lo de arriba no coincide, es NORMAL: por eso hay controles supervisados aqui.")


---
### Paso 1 - Verificar ANTES de gastar GPU

No entrena. Construye **estudiante y profesor a la vez** para cada arquitectura, que
es la condicion real del Mean Teacher, e imprime parametros y memoria de GPU.

TransUNet duplicado es el unico riesgo que no se pudo medir en local. Si no cabe, se
sabe aqui en dos minutos en vez de en dos horas.


In [ ]:
# ============================================================
# VERIFICACION - NO ENTRENA. Tarda ~2 minutos.
# ============================================================
print("PUERTA 1: cada arquitectura se construye DOS veces (estudiante + profesor EMA)")
print("%-14s %10s %14s %14s" % ("arquitectura", "params M", "GPU tras 1", "GPU tras 2"))
_ok = {}
for _arch, _bb in (("unetpp", "efficientnet-b3"),
                   ("transunet", "R50-ViT-B_16"),
                   ("bifpn_unet", "vgg16")):
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        _s = create_model(_arch, _bb, 1).cuda()
        _m1 = torch.cuda.memory_allocated() / 1e9
        _t = create_model(_arch, _bb, 1).cuda()
        _m2 = torch.cuda.memory_allocated() / 1e9
        _p = sum(x.numel() for x in _s.parameters()) / 1e6
        print("%-14s %10.1f %12.2f GB %12.2f GB" % (_arch, _p, _m1, _m2))
        _ok[_arch] = True
        del _s, _t; torch.cuda.empty_cache()
    except Exception as _e:
        print("%-14s FALLA: %s: %s" % (_arch, type(_e).__name__, str(_e)[:90]))
        _ok[_arch] = False
assert all(_ok.values()), "alguna arquitectura no se construye. NO lanzar el Paso 2."
print("   OK: las tres construyen por duplicado\n")

print("PUERTA 2: el pool no etiquetado de r=15 existe y no esta vacio")
_pool = f"{BASE}/unlabeling_std_matched_r15/images"
_n = len(glob.glob(os.path.join(_pool, "*.png")))
print("   %s -> %d frames" % (_pool, _n))
assert _n > 1000, "el pool std_matched_r15 no esta donde se espera"
print("   OK\n")

print("PUERTA 3: una pasada de entrenamiento real, 1 paso, por arquitectura")
# Comprueba que el forward+backward funciona con la resolucion y el batch de aqui.
_cfg = get_default_config()
_cfg["img_root"] = BASE; _cfg["msk_root"] = BASE
_cfg["num_workers"] = 0; _cfg["use_semi"] = False
_tf = get_supervised_train_augmentation(_cfg)
_tr, _va, _te = build_supervised_datasets(_cfg, train_tf=_tf)
_ld = build_dataloaders(_cfg, train_ds=_tr, val_ds=_va, test_ds=_te)
_bx = next(iter(_ld["train_loader"]))
_x = _bx["image"].cuda(); _y = _bx["mask"].cuda()
print("   el loader entrega %d imagenes = %d frames x %d vistas, %s" % (
    _x.shape[0], _cfg["batch_size"], 1 + _cfg["num_augmented"], tuple(_x.shape[2:])))
assert _x.shape[0] == _cfg["batch_size"] * (1 + _cfg["num_augmented"])
for _arch, _bb in (("transunet", "R50-ViT-B_16"), ("bifpn_unet", "vgg16")):
    torch.cuda.empty_cache()
    _m = create_model(_arch, _bb, 1).cuda()
    _o = _m(_x)
    _l = torch.nn.functional.binary_cross_entropy_with_logits(_o, _y)
    _l.backward()
    print("   %-12s salida %s  loss %.4f  pico %.2f GB" % (
        _arch, tuple(_o.shape), float(_l), torch.cuda.max_memory_allocated() / 1e9))
    del _m, _o, _l; torch.cuda.empty_cache()
print("   OK\n")
print("TODO VERIFICADO. Se puede entrenar.")


---
### Paso 1b - Los parametros

Todo lo que se puede tocar esta aqui y solo aqui. `PLAN` es la lista literal de runs
en su orden de ejecucion.


In [ ]:
# ============================================================
# PARAMETROS DEL EXPERIMENTO
# ============================================================
BASE     = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
ROTULOS  = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
OUT_ROOT = f"{BASE}/runs_ssl_backbones"          # directorio NUEVO
os.environ["TRANSUNET_PRETRAINED_PATH"] = f"{BASE}/R50+ViT-B_16.npz"

# arquitectura -> backbone, leidos de los run_report de runs_final_v1
ARQUITECTURAS = {
    "transunet":  "R50-ViT-B_16",
    "bifpn_unet": "vgg16",
}

# El plan, en orden de ejecucion. La semilla 0 de las cuatro condiciones va
# primero: a las ~5,6 h ya se conoce la direccion de las dos arquitecturas.
# Cada entrada es (arquitectura, semi, semilla).
PLAN = [
    ("transunet",  True,  0),   # ~154 min
    ("transunet",  False, 0),   # ~ 85 min   control supervisado en esta sesion
    ("bifpn_unet", True,  0),   # ~ 63 min
    ("bifpn_unet", False, 0),   # ~ 34 min   control supervisado en esta sesion
    # ---- a partir de aqui, solo si queda tiempo de GPU ----
    ("transunet",  True,  1),   # ~154 min
    ("transunet",  True,  2),   # ~154 min
    ("bifpn_unet", True,  1),   # ~ 63 min
    ("bifpn_unet", True,  2),   # ~ 63 min
]

MINUTOS = {("transunet", True): 154, ("transunet", False): 85,
           ("bifpn_unet", True): 63, ("bifpn_unet", False): 34}

print("salida :", OUT_ROOT)
print("runs   : %d" % len(PLAN))
print()
print("%-3s %-12s %-12s %-8s %s" % ("#", "arquitectura", "condicion", "semilla", "acumulado"))
_ac = 0
for _i, (_a, _s, _sd) in enumerate(PLAN, 1):
    _ac += MINUTOS[(_a, _s)]
    print("%-3d %-12s %-12s %-8d %.1f h" % (_i, _a, "MT r=15" if _s else "supervisado", _sd, _ac / 60))


---
### Paso 1c - El cuerpo de un run

Una sola funcion. El bloque `cfg` copia la configuracion de
`runs_final_v1/mean_teacher_fpn_std_matched_r15`, que es una de las cuatro filas que
ya estan en la tesis. Lo unico que cambia entre runs es `arch`, `use_semi` y `seed`.


In [ ]:
def run_arm(arch, semi, semilla, verbose=False):
    """Train one backbone under Mean Teacher or supervised, and return its exp_dir.

    The configuration is the one behind the four SSL-backbone rows already reported
    in the manuscript; only the architecture, the semi-supervised switch and the seed
    change between runs. A finished run is skipped instead of being retrained.
    """
    import gc; gc.collect()
    torch.cuda.empty_cache()

    nombre = f"{'mean_teacher' if semi else 'supervised'}_{arch}_std_matched_r15"

    cfg = get_default_config()
    cfg["img_root"]    = BASE
    cfg["msk_root"]    = BASE
    cfg["rotulos_dir"] = ROTULOS
    cfg["exp_dir"]     = f"{OUT_ROOT}/{nombre}/seed_{semilla}"

    cfg["arch"]      = arch
    cfg["backbone"]  = ARQUITECTURAS[arch]
    cfg["n_classes"] = 1

    cfg["seed"] = semilla

    # --- configuracion SSL, copiada de mean_teacher_fpn_std_matched_r15 ---
    cfg["use_semi"]             = bool(semi)
    cfg["ssl_method"]           = "mean_teacher"
    cfg["lambda_u"]             = 0.05 if semi else 0.0
    cfg["tau"]                  = 0.95
    cfg["ema_decay"]            = 0.99
    cfg["semi_start_epoch"]     = 15
    cfg["semi_warmup_epochs"]   = 20
    cfg["unlabeled_subdir"]     = "unlabeling_std_matched_r15/images"
    cfg["use_temp_consistency"] = False
    cfg["lambda_t"]             = 0.0

    cfg["image_preproc"]  = "base"
    cfg["mask_smoothing"] = "none"
    cfg["use_fixed_crop"] = False
    cfg["target_size"]    = (320, 320)
    cfg["use_pad"]        = True
    cfg["imagenet_norm"]  = False

    cfg["batch_size"]     = 5
    cfg["num_workers"]    = 4
    cfg["drop_last"]      = True
    cfg["num_augmented"]  = 5
    cfg["lr"]             = 1e-3
    cfg["weight_decay"]   = 1e-4
    cfg["epochs"]         = 2000
    cfg["warmup_epochs"]  = 10
    cfg["patience_es"]    = 40
    cfg["eval_threshold"] = 0.5

    cfg["save_preds_vis"] = False
    cfg["run_ruler_eval"] = True

    _exp = cfg["exp_dir"]
    _best    = os.path.isfile(os.path.join(_exp, "best_model.pt"))
    _metrics = os.path.isfile(os.path.join(_exp, "test_metrics.csv"))
    _summary = os.path.isfile(os.path.join(_exp, "run_summary.txt"))
    _report  = len(glob.glob(os.path.join(_exp, "*_run_report.json"))) > 0

    if _best and not _summary:
        print(f"AVISO {nombre}/seed_{semilla}: best_model.pt sin run_summary.txt.")
        print("      Run cortado a medias. Se reentrena desde cero.")

    if _best and _metrics and _summary:
        print(f"Skipping {nombre}/seed_{semilla}: run completo detectado")
        return _exp

    if verbose:
        print(summarize_config(cfg))

    # guardian en caliente: lo que distingue a este run tiene que estar en el cfg
    assert cfg["arch"] == arch, f"FALLO: cfg[arch] es {cfg['arch']!r}, no {arch!r}"
    assert cfg["use_semi"] == bool(semi), "FALLO: use_semi no coincide"
    assert cfg["seed"] == semilla, "FALLO: seed no coincide"
    print(f"guardian OK: {nombre} seed={semilla} arch={arch} use_semi={bool(semi)}")

    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    # El pool sin etiquetar hay que construirlo aqui: build_dataloaders no lo crea,
    # solo envuelve el que se le pase. Sin esto unlabeled_loader queda en None y la
    # rama SSL no se ejecuta aunque el cfg diga use_semi=True.
    unlabeled_ds, temporal_unlab_ds = None, None
    if cfg.get("use_semi", False) or cfg.get("use_temp_consistency", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg)
    loaders = build_dataloaders(cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)

    # Guardian del EFECTO, no de la intencion. El 3 y 4 de septiembre diecinueve
    # runs se declararon Mean Teacher y entrenaron sin una imagen sin etiquetar.
    if cfg.get("use_semi", False):
        assert loaders.get("unlabeled_loader") is not None, (
            "FALLO: use_semi=True pero no se construyo el unlabeled_loader")
        assert len(loaders["unlabeled_loader"].dataset) > 0, (
            "FALLO: el pool sin etiquetar esta vacio")
        print(f"pool OK: {len(loaders['unlabeled_loader'].dataset)} frames sin "
              f"etiquetar de {cfg['unlabeled_subdir']}")

    # guardian en caliente 2: el loader de train aplana 1 vista base + num_augmented
    # vistas por frame (SegmentationDataset.__getitem__ + flatten_collate), de modo
    # que el tensor trae batch_size * (1 + num_augmented) imagenes, no batch_size.
    _bx = next(iter(loaders["train_loader"]))["image"]
    _esperado = cfg["batch_size"] * (1 + cfg["num_augmented"])
    assert _bx.shape[0] == _esperado, (
        f"FALLO: el loader entrega {_bx.shape[0]} imagenes, no {_esperado} = "
        f"batch_size {cfg['batch_size']} x (1 + num_augmented {cfg['num_augmented']}).")
    assert tuple(_bx.shape[2:]) == tuple(cfg["target_size"]), "FALLO: resolucion inesperada"
    print(f"loader OK: {_bx.shape[0]} imagenes por paso = "
          f"{cfg['batch_size']} frames x {1 + cfg['num_augmented']} vistas, {tuple(_bx.shape[2:])}")

    # Copia de todo lo que imprime el run, sin dejar de mostrarlo en pantalla.
    with tee_output(os.path.join(_exp, "stdout.log")):
        _t0 = time.time()
        if _best and _summary and (not _metrics or not _report):
            _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _m, loaders,
                                          os.path.join(_exp, "best_model.pt"), [])
        else:
            art = run_training(cfg, loaders)
            results = evaluate_checkpoint(cfg, art["model"], loaders,
                                          art["best_path"], art["history"])
            # Tercer guardian, con el run ya terminado: la perdida no supervisada
            # tiene que haberse movido. Con lambda_u=0 sigue siendo distinta de cero,
            # asi que un cero solo puede significar que la rama no se ejecuto.
            if cfg.get("use_semi", False):
                _mu = max([(e.get("unsup_loss") or 0.0)
                           for e in (art["history"] or [])] or [0.0])
                assert _mu > 0, (
                    "FALLO: unsup_loss se quedo en cero en todas las epocas; "
                    "el run NO fue semi-supervisado")
        print(f"[{nombre}/seed_{semilla}] {(time.time()-_t0)/60:.1f} min")
        print(results)
    return _exp


print("run_arm definida.")


---
## Paso 2 - Los runs

La semilla 0 de las cuatro condiciones va primero. **Se puede cortar en cualquier
momento**: al reejecutar la celda, los runs terminados se saltan y sigue por donde
iba. Los cuatro primeros son ~5,6 h; los ocho, ~12,8 h.


In [ ]:
# === PASO 2: el plan, en orden ===
_fallos = []
for _i, (_arch, _semi, _seed) in enumerate(PLAN, 1):
    _cond = "MT r=15" if _semi else "supervisado"
    print("=" * 70)
    print(f">>> [{_i}/{len(PLAN)}]  {_arch}  {_cond}  seed={_seed}"
          f"   (~{MINUTOS[(_arch, _semi)]} min)")
    print("=" * 70)
    try:
        run_arm(_arch, _semi, _seed, verbose=False)
    except Exception as e:
        print(f"FALLO en {_arch}/{_cond}/seed_{_seed}: {type(e).__name__}: {e}")
        _fallos.append((_arch, _cond, _seed, repr(e)))

print()
print("terminado. fallos:", len(_fallos))
for f in _fallos:
    print("  ", f)


---
## Paso 3 - El resumen

Solo lectura. Compara cada arquitectura contra su **propio control supervisado
entrenado en esta misma sesion**, y contra las cuatro filas que ya estan en la tesis.


In [ ]:
# === PASO 3: RESUMEN (solo lectura) ===
def leer_report(exp_dir):
    _f = glob.glob(os.path.join(exp_dir, "*_run_report.json"))
    if not _f:
        return None
    _r = json.load(open(_f[0], encoding="utf-8"))
    _tm = _r.get("test_metrics") or {}
    _bm = _r.get("test_boundary_metrics") or {}
    return {"f1": _tm.get("sample_mean_f1"), "bf1": _bm.get("bf1_mean"),
            "assd": _bm.get("assd_mean_px"), "hd95": _bm.get("hd95_mean_px")}


def agregar(arch, semi):
    _n = f"{'mean_teacher' if semi else 'supervised'}_{arch}_std_matched_r15"
    _v = []
    for _s in (0, 1, 2):
        _d = leer_report(f"{OUT_ROOT}/{_n}/seed_{_s}")
        if _d and _d["f1"] is not None:
            _v.append(_d)
    if not _v:
        return None
    _f1 = [d["f1"] for d in _v]
    return {"n": len(_v), "f1": statistics.mean(_f1),
            "sd": statistics.stdev(_f1) if len(_f1) > 1 else 0.0,
            "crudos": _f1,
            "bf1": statistics.mean([d["bf1"] for d in _v if d["bf1"] is not None] or [float("nan")])}


print("RESULTADOS DE ESTA SESION")
print("%-14s %-12s %-4s %-18s %-8s %s" % ("arquitectura", "condicion", "n", "F1", "BF1", "crudos"))
_res = {}
for _arch in ARQUITECTURAS:
    for _semi in (False, True):
        _a = agregar(_arch, _semi)
        _res[(_arch, _semi)] = _a
        if _a is None:
            print("%-14s %-12s   -   sin resultado" % (_arch, "MT r=15" if _semi else "supervisado"))
            continue
        print("%-14s %-12s %-4d %.4f +/- %.4f  %.3f    %s" % (
            _arch, "MT r=15" if _semi else "supervisado", _a["n"], _a["f1"], _a["sd"],
            _a["bf1"], " ".join("%.4f" % x for x in _a["crudos"])))

print()
print("EFECTO DEL SSL, contra el control supervisado DE ESTA SESION")
for _arch in ARQUITECTURAS:
    _s, _m = _res[(_arch, False)], _res[(_arch, True)]
    if _s and _m:
        print("   %-14s %+.4f   (%.4f -> %.4f)" % (_arch, _m["f1"] - _s["f1"], _s["f1"], _m["f1"]))
    else:
        print("   %-14s incompleto" % _arch)

print()
print("LAS CUATRO FILAS QUE YA ESTAN EN LA TESIS (l.915), para contexto:")
print("   U-Net++      .801 -> .830 +/- .038   (+.029)")
print("   U-Net        .824 -> .851 +/- .005   (+.027)")
print("   FPN          .799 -> .811 +/- .028   (+.012)")
print("   DeepLabV3+   .776 -> .800 +/- .014   (+.024)")
print()
print("OJO: esos cuatro se entrenaron con el stack de julio. El control supervisado")
print("de arriba es el que hay que usar para juzgar el efecto en estas dos.")
